# summary -> training, generation and judged results from every run, in the tables and plots the report uses

## imports

In [ ]:
from pathlib import Path
from math import comb
import json
import random
import re

import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

## global env variables ->

In [ ]:
#everything here reads the results already pulled back into the repo, no gpu and no judge call is needed. run it from
#this folder, the paths are relative to it
TRAIN_DIR = '../train/modal_run/outputs/train'
GENERATE_DIR = '../generate/modal_run/outputs'
PERSONA_DIR = '../persona/modal_run/outputs/persona'
JUDGE_DIR = '../judge/kaggle_run/outputs/judge'
OUTPUT_DIR = './outputs/summary'

TOKENIZER = 'Qwen/Qwen2.5-0.5B' #only used to count answer tokens, so a loop can be told apart from a long answer
MAX_NEW_TOKENS = 384 #the generation limit every run used
BOOTSTRAP_SAMPLES = 10000
SEED = 42

#the decoding runs, in the order they are reported. each judge folder holds the verdicts for one decoding
DECODINGS = {'greedy': ['generate', 'generate_dpo_uf_lr5e-6'],
             'rep1.1': ['generate_rep1.1', 'generate_rep1.1_dpo_uf_lr5e-6']}
PERSONA_DECODINGS = {'persona': 'generate', 'persona_rep1.1': 'generate_rep1.1'}

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


#an answer that used up the whole budget and whose tail is one stretch of text repeated at least three times. the
#budget condition keeps a short answer that ends on a repeated sign off from counting as a loop
TAIL_LOOP = re.compile(r'(.{4,300}?)(?:\s*\1){2,}\s*$', re.S)


def is_loop(text):
    return len(tokenizer.encode(text)) >= MAX_NEW_TOKENS - 10 and TAIL_LOOP.search(text[-1500:]) is not None

## training -> one row per run, the checkpoint each later stage used

In [ ]:
training = []
#pipeline order -> sft first, then the dpo runs that start from it
for done_path in sorted(Path(TRAIN_DIR).glob('*_done.json'), key=lambda path: (not path.name.startswith('sft'), path.name)):
    done = json.loads(done_path.read_text())
    history = pd.read_csv(Path(TRAIN_DIR) / f'{done["run"]}_history.csv')
    validation = history.dropna(subset=[column for column in history if column.startswith('val_')][:1])
    row = {'run': done['run'], 'steps': done['steps'], 'best_step': done['best_step'],
           'minutes': done['seconds'] / 60, 'reference': done.get('reference_checkpoint', '')}
    if 'val_reward_accuracy' in history:
        #the dpo runs pick their checkpoint by validation reward accuracy, the margin is read at that same step
        best = validation.loc[validation['step'] == done['best_step']].iloc[-1]
        row.update(best_val_reward_accuracy=best['val_reward_accuracy'], best_val_reward_margin=best['val_reward_margin'],
                   best_val_loss=best['val_loss'], start_val_loss=validation['val_loss'].iloc[0])
    else:
        row.update(best_val_loss=done['best_val_loss'], start_val_loss=validation['val_loss'].iloc[0])
    training.append(row)

#the persona runs, from the trl logs. same columns where they mean the same thing
selection = pd.read_csv(Path(PERSONA_DIR) / 'generate' / 'selection.csv')
for _, run in selection.iterrows():
    training.append({'run': run['run'], 'steps': run['steps'], 'best_step': run['best_step'],
                     'minutes': run['seconds'] / 60, 'reference': 'SmolLM2-360M-Instruct (adapter disabled)',
                     'best_val_reward_accuracy': run['best_eval_reward_accuracy'],
                     'best_val_reward_margin': run['best_eval_reward_margin'], 'best_val_loss': run['best_eval_loss']})

training = pd.DataFrame(training)
training.to_csv(Path(OUTPUT_DIR) / 'training_summary.csv', index=False)
display(training.round(3))

## generation -> how often each model stops on its own, how long it talks, how often greedy decoding loops

In [ ]:
generation = []
answers = {} #(decoding, generator, eval set) -> {prompt id: answer}, read again for the judged results below
for decoding, folders in DECODINGS.items():
    for folder in folders:
        stats = pd.read_csv(Path(GENERATE_DIR) / folder / 'generation_summary.csv')
        for _, row in stats.iterrows():
            key = (decoding, row['generator'], row['eval_set'])
            if key in answers:
                continue #sft is generated again next to each retry, byte identical, so it is counted once
            rows = json.loads((Path(GENERATE_DIR) / folder / f'{row["generator"]}__{row["eval_set"]}.json').read_text())
            answers[key] = {entry['id']: entry['output'] for entry in rows}
            generation.append({'decoding': decoding, 'generator': row['generator'], 'eval_set': row['eval_set'],
                               'prompts': row['prompts'], 'stops_on_its_own': row['reached_eos'],
                               'mean_tokens': row['mean_tokens'],
                               'loops': sum(is_loop(output) for output in answers[key].values())})

generation = pd.DataFrame(generation)
generation.to_csv(Path(OUTPUT_DIR) / 'generation_summary.csv', index=False)
display(generation.round(3).set_index(['decoding', 'eval_set', 'generator']))

## dpo vs sft -> the headline comparison, with how sure each number is

In [ ]:
#two sided sign test on the decisive pairs -> how likely a split at least this uneven is if neither model is better
def sign_test(wins, losses):
    n, k = int(wins + losses), int(min(wins, losses)) #python ints, 2 ** n overflows a numpy integer
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0


#95% interval for the win rate with ties as half a win, by resampling prompts with replacement
def bootstrap_interval(scores):
    rng = random.Random(SEED)
    means = sorted(sum(rng.choices(scores, k=len(scores))) / len(scores) for _ in range(BOOTSTRAP_SAMPLES))
    return means[int(0.025 * BOOTSTRAP_SAMPLES)], means[int(0.975 * BOOTSTRAP_SAMPLES) - 1]


judge_summary = pd.read_csv(Path(JUDGE_DIR) / 'judge_summary.csv')
headline, judged_rows = [], []
for decoding in DECODINGS:
    #hh first, then uf and its retry, so each retry row sits right under the run it repeats
    order = lambda path: (not path.name.startswith('dpo_hh'), len(path.name), path.name)
    for judged_path in sorted((Path(JUDGE_DIR) / decoding).glob('*_judged.jsonl'), key=order):
        rows = read_jsonl(judged_path)
        challenger, baseline = judged_path.name.replace('_judged.jsonl', '').split('__')[0].split('_vs_')
        eval_set = rows[0]['dataset']
        for row in rows:
            row.update(decoding=decoding, challenger=challenger,
                       score=1.0 if row['winner_model'] == challenger else 0.0 if row['winner_model'] == baseline else 0.5,
                       #a loop on either side, read from the answers rather than from the pair file
                       loop=any(is_loop(answers[(decoding, model, eval_set)][row['id']]) for model in (challenger, baseline)))
        judged_rows += rows

        counts = judge_summary.query('run == @decoding and challenger == @challenger').iloc[0]
        low, high = bootstrap_interval([row['score'] for row in rows])
        clean = [row for row in rows if not row['loop']]
        headline.append({
            'decoding': decoding, 'comparison': f'{challenger} vs {baseline}', 'eval_set': eval_set,
            'pairs': counts['pairs'], 'wins': counts['wins'], 'losses': counts['losses'], 'ties': counts['ties'],
            'unknown': counts['unknown'], 'failed': counts['failed'], 'retries': counts['retries'],
            'win_rate_with_ties': counts['win_rate_with_ties'], 'ci_low': low, 'ci_high': high,
            'win_rate_without_ties': counts['win_rate_without_ties'],
            'sign_test_p': sign_test(counts['wins'], counts['losses']),
            'picked_a': counts['picked_a'],
            #the same rate on pairs where neither answer loops, so looping alone cannot explain the result
            'loop_free_pairs': len(clean),
            'loop_free_win_rate': sum(row['score'] for row in clean) / max(len(clean), 1),
            'judge_model': counts['judge_model'],
        })

headline = pd.DataFrame(headline)
headline.to_csv(Path(OUTPUT_DIR) / 'dpo_vs_sft.csv', index=False)
display(headline.drop(columns='judge_model').round(3).set_index(['decoding', 'comparison']))
print('judge:', ', '.join(headline['judge_model'].unique()))

In [ ]:
#wins, ties and losses per comparison as one stacked bar each, decodings side by side. ties sit in the middle so the
#win and loss shares can be compared from either end
fig, ax = plt.subplots(figsize=(9, 0.55 * len(headline) + 1.2))
labels = [f'{row.comparison}  ({row.eval_set}, {row.decoding})' for row in headline.itertuples()]
left = 0
for column, colour in [('wins', '#2e7d32'), ('ties', '#bdbdbd'), ('losses', '#c62828')]:
    share = headline[column] / headline['pairs']
    ax.barh(labels, share, left=left, color=colour, label=column)
    left = left + share
ax.axvline(0.5, color='black', linewidth=0.8, linestyle='--')
for index, row in enumerate(headline.itertuples()):
    ax.text(1.01, index, f'{row.win_rate_with_ties:.1%} [{row.ci_low:.0%}, {row.ci_high:.0%}]', va='center', fontsize=8)
ax.set_xlim(0, 1)
ax.set_xlabel(f'share of pairs from the dpo side, judge {headline["judge_model"].iloc[0]}\n'
              'right: win rate with ties as half a win, and its 95% bootstrap interval')
ax.invert_yaxis()
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, frameon=False)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR) / 'dpo_vs_sft.png', dpi=200, bbox_inches='tight')
plt.show()

## where the hh result comes from -> per category of the helpful / harmless set

In [ ]:
frame = pd.DataFrame(judged_rows)
hh = frame[frame['dataset'] == 'hh_eval'].copy()
hh['outcome'] = hh['score'].map({1.0: 'win', 0.5: 'tie', 0.0: 'loss'})
by_category = (hh.groupby(['category', 'decoding', 'outcome']).size().unstack(['decoding', 'outcome'], fill_value=0)
               .reindex(columns=pd.MultiIndex.from_product([list(DECODINGS), ['win', 'tie', 'loss']]), fill_value=0))
for decoding in DECODINGS:
    by_category[(decoding, 'net')] = by_category[(decoding, 'win')] - by_category[(decoding, 'loss')]
by_category = by_category.sort_index(axis=1, level=0, sort_remaining=False)
by_category.to_csv(Path(OUTPUT_DIR) / 'dpo_hh_by_category.csv')
display(by_category)

## persona training -> every lora run, the step each one kept, and the adapter each decoding selected

In [ ]:
#the selection is made on validation answers, so it depends on how they were decoded. both choices are marked
selected_by = {}
for decoding, folder in PERSONA_DECODINGS.items():
    run_config = json.loads((Path(PERSONA_DIR) / folder / 'run_config.json').read_text())
    selected_by.setdefault(run_config['dpo']['selected_run'], []).append(decoding)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
for row_axes, (rpo_alpha, setting) in zip(axes, [(0.0, 'plain dpo'), (1.0, 'rpo alpha 1')]):
    for run in selection[selection['rpo_alpha'] == rpo_alpha].itertuples():
        history = pd.read_csv(Path(PERSONA_DIR) / f'{run.run}_history.csv')
        train_rows, val_rows = history.dropna(subset=['loss']), history.dropna(subset=['eval_loss'])
        kept = val_rows[val_rows['step'] == run.best_step].iloc[-1] #the step whose adapter was saved and used
        chosen = selected_by.get(run.run)
        label = f'lr {run.learning_rate:g}' + (f'  <- selected ({", ".join(chosen)})' if chosen else '')
        for axis, metric in zip(row_axes, ['loss', 'rewards/accuracies', 'rewards/margins']):
            line, = axis.plot(train_rows['step'], train_rows[metric], alpha=0.35)
            axis.plot(val_rows['step'], val_rows[f'eval_{metric}'], marker='o', markersize=3, color=line.get_color(),
                      linewidth=2.5 if chosen else 1.2, label=label)
            axis.plot(kept['step'], kept[f'eval_{metric}'], marker='*', markersize=14, color=line.get_color(),
                      markeredgecolor='black', linestyle='none')
    for axis, label in zip(row_axes, ['loss', 'reward accuracy', 'reward margin']):
        axis.set_xlabel('optimizer step')
        axis.set_ylabel(label)
        axis.set_title(f'persona lora, {setting}: {label}', fontsize=10)
    row_axes[0].legend(fontsize=8, title='validation (thick = selected)', title_fontsize=8)
fig.suptitle(f'faint: train, mean of 5 steps. dots: validation. star: the checkpoint each run kept (lowest validation '
             f'loss). rpo losses include the sft term, so their scale differs', fontsize=9)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR) / 'persona_training.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
#the automatic style and length checks, base against the adapter each decoding selected, per prompt category
persona_generation = []
for decoding, folder in PERSONA_DECODINGS.items():
    stats = pd.read_csv(Path(PERSONA_DIR) / folder / 'generation_summary.csv')
    selected_run = next(run for run, decodings in selected_by.items() if decoding in decodings)
    persona_generation.append(stats[stats['generator'].isin(['smollm2_base', selected_run])].assign(decoding=decoding))
persona_generation = pd.concat(persona_generation)[['decoding', 'generator', 'category', 'reached_eos', 'mean_tokens',
                                                    'exclamations_per_answer', 'insult_rate']]
persona_generation.to_csv(Path(OUTPUT_DIR) / 'persona_generation.csv', index=False)
display(persona_generation.round(3).set_index(['decoding', 'generator', 'category']))

## persona -> each adapter against the base model, style and quality kept apart

In [ ]:
persona = []
for judge_run, folder in PERSONA_DECODINGS.items():
    judged_paths = sorted((Path(JUDGE_DIR) / judge_run).glob('*_judged.jsonl'))
    stats = pd.read_csv(Path(PERSONA_DIR) / folder / 'generation_summary.csv').query('category == "all"')
    for judged_path in judged_paths:
        rows = read_jsonl(judged_path)
        adapter = judged_path.name.split('_vs_')[0]
        verdicts = pd.DataFrame(rows)

        def rate(column, part=verdicts):
            #ties as half a win, from the adapter's side
            scores = part[column].map(lambda model: 1.0 if model == adapter else 0.0 if isinstance(model, str) else 0.5)
            return scores.mean()

        safety = verdicts[verdicts['category'] == 'safety_sensitive']
        generated = stats.query('generator == @adapter').iloc[0]
        persona.append({'decoding': judge_run, 'adapter': adapter, 'selected': generated['selected'],
                        'pairs': len(rows), 'style_win_rate': rate('style_winner_model'),
                        'quality_win_rate': rate('winner_model'),
                        'safety_quality_win_rate': rate('winner_model', safety),
                        'unknown': int((verdicts['verdict'] == 'unknown').sum()),
                        'stops_on_its_own': generated['reached_eos'], 'mean_tokens': generated['mean_tokens'],
                        'exclamations_per_answer': generated['exclamations_per_answer']})

persona = pd.DataFrame(persona)
if len(persona):
    persona.to_csv(Path(OUTPUT_DIR) / 'persona_vs_base.csv', index=False)
    display(persona.round(3).set_index(['decoding', 'adapter']))
else:
    print(f'no persona verdicts in {JUDGE_DIR} yet')

In [ ]:
#style against quality, one point per adapter. top right is the goal: more persona without being less useful or safe
if len(persona):
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for decoding, marker in zip(PERSONA_DECODINGS, ['o', 's']):
        part = persona[persona['decoding'] == decoding]
        ax.scatter(part['style_win_rate'], part['quality_win_rate'], marker=marker, s=45, label=decoding,
                   facecolors=['black' if selected else 'none' for selected in part['selected']], edgecolors='black')
        for row in part.itertuples():
            #the low rate adapters barely change the style and pile up near 0.5, only the ones that moved are named
            if row.style_win_rate > 0.6 or row.selected:
                ax.annotate(row.adapter.replace('persona_', ''), (row.style_win_rate, row.quality_win_rate),
                            fontsize=7, xytext=(4, 3), textcoords='offset points')
    ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.axvline(0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('style win rate vs base (ties as half)')
    ax.set_ylabel('quality win rate vs base (ties as half)')
    ax.set_title('persona adapters vs smollm2 base, filled = selected adapter\n'
                 'unlabelled cluster near style 0.5: the lr 1e-5 and 2e-5 adapters', fontsize=9)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(Path(OUTPUT_DIR) / 'persona_style_vs_quality.png', dpi=200, bbox_inches='tight')
    plt.show()